# Step3: Standard Downstream

Standard downstream notebook after annotation.

This notebook keeps the broadly reusable downstream analyses together:

- cell-type composition and sample-level proportions
- group comparisons of proportions
- per-celltype DEG setup/results summary
- enrichment summary across comparisons

Input: `data/processed/Step3-sce_annotated.h5ad`.


---

## 1. Setup

### 1.1 Imports


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import scLucid as scl
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
import warnings
warnings.filterwarnings('ignore')

scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={'axes.grid': True},
    color_theme="default"
)


### 1.2 Paths


In [ ]:
# Define working directories
BASE_DIR = '/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202604JJH'
DATA_DIR = os.path.join(BASE_DIR, "1-DATA")
RESULTS_DIR = os.path.join(BASE_DIR, "2-OUTPUT")
os.makedirs(RESULTS_DIR, exist_ok=True)


### 1.3 Colors


In [ ]:
###** colors
color = {
    'PBS': '#BFCFE3',
    "R301":"#8B4C9D"
}


In [ ]:
print("Loading annotated data...")
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step3-sce_annotated.h5ad"))
print(f"Dataset loaded: {adata.n_obs:,} cells, {adata.n_vars:,} genes")
if "celltype" not in adata.obs.columns:
    raise KeyError("Expected adata.obs['celltype']; run Step2-Annotation_and_Malignancy.ipynb first.")


---

## 5. Proportion Analysis

### 5.1 Load Data


In [ ]:
# adata loaded above from Step3-sce_annotated.h5ad
adata


### 5.2 Visualize


In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['celltype'],
                title = ['All cells'],
                legend_loc="on data",
                legend_fontsize=5,
                size=15,
                palette=celltype_colors,
                show=True, frameon=False,
                save=True)


### 5.3 Group Counts


In [ ]:
adata.obs['group'].value_counts()


### 5.5 Cell Count Barplot


In [ ]:
# ==============================================================================
# Cell Count Barplot by Group (5 Treatment Groups)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# Get cell type order by total count
celltype_order = adata.obs['celltype'].value_counts().index.tolist()

# Count cells per group and celltype
count_df = adata.obs.groupby(['group', 'celltype']).size().reset_index(name='count')

# Filter to existing groups
existing_groups = [g for g in treatment_order if g in count_df['group'].values]

# Plot
fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(
    data=count_df,
    x='celltype',
    y='count',
    order=celltype_order,
    hue='group',
    hue_order=existing_groups,
    palette=color,
    ax=ax
)

# Add count labels on bars
for patch in ax.patches:
    height = patch.get_height()
    if height > 0 and not pd.isna(height):
        ax.text(
            x=patch.get_x() + patch.get_width() / 2,
            y=height,
            s=f'{int(height)}',
            ha='center',
            va='bottom',
            fontsize=8,
            rotation=90
        )

ax.set_xlabel("")
ax.set_title('Cell Counts per Cell Type by Treatment Group', fontsize=14)
ax.set_ylabel('Cell count', fontsize=12)
plt.xticks(rotation=45, ha='right')
ax.legend(title='Group', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_Barplot_cell_counts_by_group.pdf'),
    bbox_inches='tight',
    dpi=300
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_Barplot_cell_counts_by_group.pdf')}")


### 5.6 Stacked Barplot - Proportion


In [ ]:
# ==============================================================================
# Stacked Barplot - Cell Proportion BY GROUP (3 Treatment Groups)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# Calculate proportions per sample
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group info
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

# Calculate GROUP means (not sample-level)
group_props = proportions_df.groupby('group')[proportions_df.columns[:-1]].mean()

# Reindex to treatment order
existing_groups = [g for g in treatment_order if g in group_props.index]
group_props = group_props.reindex(existing_groups)

# Define celltype colors
if 'celltype_colors' not in globals():
    import matplotlib.cm as cm
    celltypes = group_props.columns
    colors = cm.tab20.colors[:len(celltypes)]
    celltype_colors_local = dict(zip(celltypes, colors))
else:
    celltype_colors_local = celltype_colors

plot_colors = [celltype_colors_local.get(ct, 'gray') for ct in group_props.columns]

# Plot stacked barplot BY GROUP
fig, ax = plt.subplots(figsize=(8, 7))
group_props.plot(
    kind='bar',
    stacked=True,
    color=plot_colors,
    ax=ax,
    edgecolor='white',
    linewidth=0.5
)

ax.set_ylabel('Cell Proportion', fontsize=12)
ax.set_xlabel('Treatment Group', fontsize=12)
ax.set_title('Cell Type Composition by Treatment Group', fontsize=14)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Cell Type', fontsize=9)

# Rotate x labels
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_StackedBar_cell_proportion_BY_GROUP.pdf'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_StackedBar_cell_proportion_BY_GROUP.pdf')}")

# Print the data for reference
print("\nGroup proportions:")
print(group_props.round(4))


### 5.7 Stacked Barplot - TME


In [ ]:
# ==============================================================================
# Stacked Barplot - Malignant Cell State Composition BY GROUP
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import os

prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# Define malignant cell states
# ------------------------------------------------------------------------------

malignant_celltypes = [
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells"
]

# Keep only malignant cell types that actually exist in adata.obs['celltype']
existing_malignant_celltypes = [
    ct for ct in malignant_celltypes
    if ct in adata.obs["celltype"].unique()
]

if len(existing_malignant_celltypes) == 0:
    raise ValueError("No malignant cell types found in adata.obs['celltype']. Please check celltype names.")

print("Malignant cell types used:")
print(existing_malignant_celltypes)

# ------------------------------------------------------------------------------
# Calculate malignant cell state proportions per sample
# ------------------------------------------------------------------------------

# Count cells by sample and cell type
counts_df = pd.crosstab(adata.obs["sampleID"], adata.obs["celltype"])

# Select malignant cell states only
counts_malignant = counts_df.reindex(columns=existing_malignant_celltypes, fill_value=0)

# Normalize within malignant compartment per sample
# Each sample sums to 1 across malignant states
sample_totals = counts_malignant.sum(axis=1)

# Avoid division by zero if a sample has no malignant cells
proportions_df = counts_malignant.div(sample_totals.replace(0, pd.NA), axis=0)

# Add group information
sample_to_group = (
    adata.obs[["sampleID", "group"]]
    .drop_duplicates()
    .set_index("sampleID")
)

proportions_df = proportions_df.join(sample_to_group)

# Remove samples without malignant cells, if any
proportions_df = proportions_df.dropna(subset=existing_malignant_celltypes, how="all")

# ------------------------------------------------------------------------------
# Calculate group means
# ------------------------------------------------------------------------------

group_props = (
    proportions_df
    .groupby("group")[existing_malignant_celltypes]
    .mean()
)

# Reindex treatment groups
existing_groups = [g for g in treatment_order if g in group_props.index]
group_props = group_props.reindex(existing_groups)

# ------------------------------------------------------------------------------
# Define colors
# ------------------------------------------------------------------------------

if "celltype_colors" not in globals():
    import matplotlib.cm as cm
    colors = cm.tab20.colors[:len(existing_malignant_celltypes)]
    celltype_colors_local = dict(zip(existing_malignant_celltypes, colors))
else:
    celltype_colors_local = celltype_colors

plot_colors = [
    celltype_colors_local.get(ct, "gray")
    for ct in group_props.columns
]

# ------------------------------------------------------------------------------
# Plot malignant-only composition by group
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 7))

group_props.plot(
    kind="bar",
    stacked=True,
    color=plot_colors,
    ax=ax,
    edgecolor="white",
    linewidth=0.5
)

ax.set_ylabel("Proportion within malignant cells", fontsize=12)
ax.set_xlabel("Treatment Group", fontsize=12)
ax.set_title(
    "Malignant Cell State Composition by Group",
    fontsize=14,
    fontweight="bold"
)

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Malignant Cell State",
    fontsize=9
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

out_file = os.path.join(
    prop_fig_dir,
    "Fig_StackedBar_MalignantStates_BY_GROUP.pdf"
)

plt.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {out_file}")

print("\nMalignant cell state proportions by group:")
print(group_props.round(4))


### 5.8 Alluvial Plot


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os

def plot_alluvial_single_ax(group_props, celltypes, colors_dict,
                            title, outpath=None, figsize=(12, 7),
                            bar_width=0.35, band_alpha=0.28):

    groups = list(group_props.index)
    n = len(groups)
    x = np.arange(n)

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # 计算每个 group 的堆叠起点 bottom
    bottoms = {g: 0.0 for g in groups}

    # 为了能画“带子”，我们需要存每个 celltype 在每个 group 的 (y0,y1)
    yspans = {g: {} for g in groups}

    # 先画所有堆叠柱，并记录区间
    for ct in celltypes:
        for i, g in enumerate(groups):
            h = float(group_props.loc[g, ct]) if ct in group_props.columns else 0.0
            y0 = bottoms[g]
            y1 = y0 + h
            yspans[g][ct] = (y0, y1)

            ax.bar(x[i], h, bottom=y0, width=bar_width,
                   color=colors_dict.get(ct, "gray"),
                   edgecolor="white", linewidth=0.6)

            bottoms[g] = y1

    # 再画相邻组之间的连接带（全部画在同一个 ax 上）
    for i in range(n - 1):
        g1, g2 = groups[i], groups[i + 1]
        x_left = x[i] + bar_width / 2
        x_right = x[i + 1] - bar_width / 2

        for ct in celltypes:
            y0_l, y1_l = yspans[g1][ct]
            y0_r, y1_r = yspans[g2][ct]
            if (y1_l - y0_l) <= 0 and (y1_r - y0_r) <= 0:
                continue

            verts = [
                (x_left,  y0_l),
                (x_right, y0_r),
                (x_right, y1_r),
                (x_left,  y1_l),
            ]
            poly = patches.Polygon(
                verts, closed=True,
                facecolor=colors_dict.get(ct, "gray"),
                edgecolor="none",
                alpha=band_alpha
            )
            ax.add_patch(poly)

    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=45, ha="right")
    ax.set_xlim(-0.6, n - 1 + 0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Proportion")
    ax.set_title(title)

    # legend（单独用patch做，避免bar重复太多）
    handles = [patches.Patch(color=colors_dict.get(ct, "gray"), label=ct) for ct in celltypes]
    ax.legend(handles=handles, title="Cell Type",
              loc="center left", bbox_to_anchor=(1.02, 0.5),
              fontsize=8, title_fontsize=9, frameon=False)

    plt.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
print("Generating Alluvial Plot (BY GROUP) - ALL cell types...")

# Calculate proportions per sample then average by group
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group and calculate GROUP means
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

celltype_list_all = celltype_order
celltype_list_all = [ct for ct in celltype_list_all if ct in proportions_df.columns]

# Calculate GROUP-level proportions
group_props_all = proportions_df.groupby('group')[celltype_list_all].mean()
group_props_all = group_props_all.reindex([g for g in treatment_order 
                                           if g in group_props_all.index])

# 用你现成的 group_props_all / celltype_list_all / celltype_colors_local
outpath = os.path.join(prop_fig_dir, "Fig_Alluvial_BY_GROUP_ALL_cells.pdf")
plot_alluvial_single_ax(
    group_props=group_props_all,
    celltypes=celltype_list_all,
    colors_dict=celltype_colors_local,
    title="Cell Type Proportion Shifts by Group (ALL Cells)",
    outpath=outpath,
    figsize=(8, 6)
)
print(f"Saved: {outpath}")


In [ ]:
print("\nGenerating Alluvial Plot (BY GROUP) - TME only (no tumor)...")

# Exclude tumor and recalculate
counts_no_tumor = counts_df.drop(columns=['Malignant cells'], errors='ignore')
proportions_tme = counts_no_tumor.div(counts_no_tumor.sum(axis=1), axis=0)
proportions_tme = proportions_tme.join(sample_to_group)

celltype_list_tme = ['Monocytes', 'Mono-to-macro transitional cells',
    'Resident macrophages', 'Activated macrophages','ISG+ myeloid cells','Proliferating myeloid cells',
    'cDCs','pDCs',
    'Activated T cells','Exhausted-like T cells','NK cells',
    'Fibroblasts', 'Endothelial cells',]
celltype_list_tme = [ct for ct in celltype_list_tme if ct in proportions_tme.columns]

# Calculate GROUP-level TME proportions
group_props_tme = proportions_tme.groupby('group')[celltype_list_tme].mean()
group_props_tme = group_props_tme.reindex([g for g in treatment_order 
                                          if g in group_props_tme.index])
outpath = os.path.join(prop_fig_dir, "Fig_Alluvial_BY_GROUP_TME_no_tumor.pdf")
plot_alluvial_single_ax(
    group_props=group_props_tme,
    celltypes=celltype_list_tme,
    colors_dict=celltype_colors_local,
    title="Cell Type Proportion Shifts by Group (TME, Tumor Excluded)",
    outpath=outpath,
    figsize=(8, 6)
)
print(f"Saved: {outpath}")


### 5.9 ANOVA


In [ ]:
# ==============================================================================
# 5-Group Differential Cell Proportion Analysis (sample-level)
# 1) Build sample-level celltype proportions
# 2) Global tests per cell type:
#    - ANOVA on arcsin(sqrt(p))
#    - Kruskal–Wallis on raw p
#    - BH-FDR across cell types
# 3) Post-hoc:
#    - Tukey HSD on transformed values (within-celltype adjusted p)
#    - Pairwise MWU on raw p + within-celltype correction (FDR or Bonferroni)
# 4) Per-celltype boxplot saved individually with correct group colors + Tukey stars (*,**,***)
# ==============================================================================

import os
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from scipy.stats import f_oneway, kruskal, mannwhitneyu
import statsmodels.stats.multitest as smm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# -----------------------------
# User settings (EDIT HERE)
# -----------------------------
prop_fig_dir = os.path.join(RESULTS_DIR, "al_proportion")
os.makedirs(prop_fig_dir, exist_ok=True)

treatment_groups = [
    'PBS','R301'
]

celltype_list = celltype_order

# IMPORTANT: keys must exactly match adata.obs["group"]
MIN_CELLS_PER_SAMPLE = 200         # set 0 to disable filtering
POSTHOC_CORR = "fdr_bh"            # for MWU pairs within-celltype: "fdr_bh" or "bonferroni"
RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT = None   # e.g. 0.10 or 0.05; None=always run

# Plot options
MAKE_PLOTS = True
PLOT_FMT = "pdf"                   # "pdf" or "png"
MAX_PAIRS_TO_ANNOTATE = 6          # per plot (avoid clutter)
ANNOTATE_ONLY_SIG = True           # only show p<0.05 stars
PERCENT_YAXIS = False              # show as percent
UNIFY_YLIM = False                 # same y-limit across celltypes

# -----------------------------
# Helpers
# -----------------------------
def arcsin_sqrt_transform(p):
    p = np.clip(np.asarray(p, dtype=float), 0.0, 1.0)
    return np.arcsin(np.sqrt(p))

def fdr_series(pvals: pd.Series, method="fdr_bh") -> pd.Series:
    mask = pvals.notna()
    out = pd.Series(np.nan, index=pvals.index, dtype=float)
    if mask.sum() == 0:
        return out
    _, q, _, _ = smm.multipletests(pvals[mask].values, method=method)
    out.loc[mask] = q
    return out

def safe_global_anova(group_arrays):
    all_vals = np.concatenate([np.asarray(a) for a in group_arrays if len(a) > 0])
    if len(all_vals) == 0 or np.allclose(all_vals, all_vals[0]):
        return np.nan, np.nan
    return f_oneway(*group_arrays)

def safe_global_kw(group_arrays):
    all_vals = np.concatenate([np.asarray(a) for a in group_arrays if len(a) > 0])
    if len(all_vals) == 0 or np.allclose(all_vals, all_vals[0]):
        return np.nan, np.nan
    return kruskal(*group_arrays)

def pairwise_mwu(values, groups, group_order, alternative="two-sided"):
    df = pd.DataFrame({"value": values, "group": groups})
    out_rows = []
    for g1, g2 in itertools.combinations(group_order, 2):
        v1 = df.loc[df["group"] == g1, "value"].astype(float).values
        v2 = df.loc[df["group"] == g2, "value"].astype(float).values

        if len(v1) == 0 or len(v2) == 0:
            out_rows.append({"group1": g1, "group2": g2, "n1": len(v1), "n2": len(v2),
                             "u_stat": np.nan, "p_value": np.nan})
            continue

        try:
            u, p = mannwhitneyu(v1, v2, alternative=alternative)
        except ValueError:
            u, p = np.nan, np.nan

        out_rows.append({"group1": g1, "group2": g2, "n1": len(v1), "n2": len(v2),
                         "u_stat": u, "p_value": p})
    return pd.DataFrame(out_rows)

def p_to_stars(p: float) -> str:
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def add_sig_bracket(ax, x1, x2, y, h, text, lw=1.2, fontsize=12):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], c="black", lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom",
            fontsize=fontsize, color="black", clip_on=False)

# -----------------------------
# 1) Build sample-level proportions
# -----------------------------
counts_df = pd.crosstab(adata.obs["sampleID"], adata.obs["celltype"])

sample_to_group = (
    adata.obs[["sampleID", "group"]]
    .drop_duplicates()
    .set_index("sampleID")
)

missing_group = counts_df.index.difference(sample_to_group.index)
if len(missing_group) > 0:
    raise ValueError(f"Some sampleIDs lack group annotation: {list(missing_group)[:10]}")

proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0).join(sample_to_group)
proportions_df["n_cells"] = counts_df.sum(axis=1)

# Keep only requested celltypes that exist
celltype_list = [ct for ct in celltype_list if ct in proportions_df.columns]
if len(celltype_list) == 0:
    raise ValueError("None of the requested cell types are present in adata.obs['celltype'].")

# Filter tiny samples (optional)
if MIN_CELLS_PER_SAMPLE and MIN_CELLS_PER_SAMPLE > 0:
    before = proportions_df.shape[0]
    proportions_df = proportions_df.loc[proportions_df["n_cells"] >= MIN_CELLS_PER_SAMPLE].copy()
    after = proportions_df.shape[0]
    print(f"Filtered samples by n_cells >= {MIN_CELLS_PER_SAMPLE}: {before} -> {after}")

# Group order (existing only, in desired order)
existing_groups = [g for g in treatment_groups if g in proportions_df["group"].unique()]
if len(existing_groups) < 2:
    raise ValueError("Fewer than 2 groups present after filtering; cannot compare.")

# Quick QC
print("\nQC: n_samples per group")
print(
    proportions_df.reset_index()
    .groupby("group")["sampleID"]
    .nunique()
    .reindex(existing_groups)
    .to_string()
)
print("\nQC: cells per sample summary")
print(proportions_df["n_cells"].describe().to_string())

# Ensure colors cover existing groups
missing_colors = [g for g in existing_groups if g not in color]
if missing_colors:
    raise ValueError(f"group_colors missing keys for groups: {missing_colors}")

# -----------------------------
# 2) Global tests per cell type (sample-level)
# -----------------------------
global_rows = []
for ct in celltype_list:
    raw_arrays, tr_arrays, ns = [], [], []
    for g in existing_groups:
        vals = proportions_df.loc[proportions_df["group"] == g, ct].astype(float).values
        raw_arrays.append(vals)
        tr_arrays.append(arcsin_sqrt_transform(vals))
        ns.append(len(vals))

    # require >=2 samples per group for ANOVA variance estimate
    if any(n < 2 for n in ns):
        global_rows.append({
            "cell_type": ct,
            "group_ns": ",".join(map(str, ns)),
            "anova_f": np.nan, "anova_p": np.nan,
            "kw_h": np.nan, "kw_p": np.nan,
            "note": "Skipped: some group has <2 samples"
        })
        continue

    f_stat, p_anova = safe_global_anova(tr_arrays)
    h_stat, p_kw = safe_global_kw(raw_arrays)

    global_rows.append({
        "cell_type": ct,
        "group_ns": ",".join(map(str, ns)),
        "anova_f": f_stat, "anova_p": p_anova,
        "kw_h": h_stat, "kw_p": p_kw,
        "note": ""
    })

global_df = pd.DataFrame(global_rows)
global_df["anova_q_fdr"] = fdr_series(global_df["anova_p"], method="fdr_bh")
global_df["kw_q_fdr"] = fdr_series(global_df["kw_p"], method="fdr_bh")

# Add group mean/sem (raw proportions)
group_means = proportions_df.groupby("group")[celltype_list].mean().reindex(existing_groups)
group_sems  = proportions_df.groupby("group")[celltype_list].sem().reindex(existing_groups)
for g in existing_groups:
    global_df[f"mean_{g}"] = global_df["cell_type"].map(group_means.loc[g].to_dict())
    global_df[f"sem_{g}"]  = global_df["cell_type"].map(group_sems.loc[g].to_dict())

global_df["min_q"] = np.nanmin(
    np.vstack([global_df["anova_q_fdr"].values, global_df["kw_q_fdr"].values]),
    axis=0
)
global_df = global_df.sort_values(["min_q", "cell_type"], ascending=[True, True]).reset_index(drop=True)

global_out = os.path.join(prop_fig_dir, "CellProportion_5group_GLOBAL_ANOVA_KW_FDR.csv")
global_df.to_csv(global_out, index=False)
print(f"\nSaved global results: {global_out}")

print("\nTop global results (by min_q):")
print(global_df[["cell_type", "anova_p", "anova_q_fdr", "kw_p", "kw_q_fdr", "group_ns"]].head(20).to_string(index=False))

# -----------------------------
# 3) Post-hoc A: Tukey HSD (ANOVA follow-up; already adjusted within-celltype)
# -----------------------------
tukey_rows = []
for ct in celltype_list:
    g_row = global_df.loc[global_df["cell_type"] == ct].iloc[0]

    if RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT is not None:
        if not (pd.notna(g_row["anova_q_fdr"]) and g_row["anova_q_fdr"] < RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT):
            continue

    sub = proportions_df.loc[proportions_df["group"].isin(existing_groups), ["group", ct]].dropna().copy()
    if sub.empty or sub["group"].nunique() < 2:
        continue

    sub["value_tr"] = arcsin_sqrt_transform(sub[ct].astype(float).values)

    # If no variability, skip
    if np.allclose(sub["value_tr"].values, sub["value_tr"].values[0]):
        continue

    try:
        tuk = pairwise_tukeyhsd(endog=sub["value_tr"].values, groups=sub["group"].values, alpha=0.05)
        tdf = pd.DataFrame(tuk.summary().data[1:], columns=tuk.summary().data[0])
        for _, r in tdf.iterrows():
            tukey_rows.append({
                "cell_type": ct,
                "group1": r["group1"],
                "group2": r["group2"],
                "meandiff_tr": float(r["meandiff"]),
                "p_adj_tukey": float(r["p-adj"]),
                "lower_tr": float(r["lower"]),
                "upper_tr": float(r["upper"]),
                "reject_0p05": bool(r["reject"]),
                "global_anova_p": g_row["anova_p"],
                "global_anova_q_fdr": g_row["anova_q_fdr"],
            })
    except Exception as e:
        tukey_rows.append({
            "cell_type": ct, "group1": None, "group2": None,
            "meandiff_tr": np.nan, "p_adj_tukey": np.nan,
            "lower_tr": np.nan, "upper_tr": np.nan, "reject_0p05": False,
            "global_anova_p": g_row["anova_p"],
            "global_anova_q_fdr": g_row["anova_q_fdr"],
            "error": str(e)
        })

tukey_df = pd.DataFrame(tukey_rows)
tukey_out = os.path.join(prop_fig_dir, "CellProportion_5group_POSTHOC_TukeyHSD_arcsin.csv")
tukey_df.to_csv(tukey_out, index=False)
print(f"Saved Tukey post-hoc: {tukey_out}")

# -----------------------------
# 3) Post-hoc B: Pairwise MWU (KW follow-up) + within-celltype correction
# -----------------------------
mwu_blocks = []
for ct in celltype_list:
    g_row = global_df.loc[global_df["cell_type"] == ct].iloc[0]

    if RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT is not None:
        if not (pd.notna(g_row["kw_q_fdr"]) and g_row["kw_q_fdr"] < RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT):
            continue

    sub = proportions_df.loc[proportions_df["group"].isin(existing_groups), ["group", ct]].dropna().copy()
    if sub.empty or sub["group"].nunique() < 2:
        continue

    pw = pairwise_mwu(
        values=sub[ct].astype(float).values,
        groups=sub["group"].values,
        group_order=existing_groups,
        alternative="two-sided"
    )
    pw["p_corr"] = fdr_series(pw["p_value"], method=POSTHOC_CORR).values
    pw["reject_corr_0p05"] = pw["p_corr"] < 0.05
    pw.insert(0, "cell_type", ct)
    pw["global_kw_p"] = g_row["kw_p"]
    pw["global_kw_q_fdr"] = g_row["kw_q_fdr"]
    pw["corr_method_within_celltype"] = POSTHOC_CORR
    mwu_blocks.append(pw)

mwu_df = pd.concat(mwu_blocks, ignore_index=True) if len(mwu_blocks) else pd.DataFrame()
mwu_out = os.path.join(prop_fig_dir, f"CellProportion_5group_POSTHOC_MWU_withinCelltype_{POSTHOC_CORR}.csv")
mwu_df.to_csv(mwu_out, index=False)
print(f"Saved MWU post-hoc: {mwu_out}")

# -----------------------------
# 4) Plot per celltype with improved typography/layout + Tukey star annotations
# -----------------------------
if MAKE_PLOTS:
    plot_dir = os.path.join(prop_fig_dir, "celltype_proportion_plots_annotated")
    os.makedirs(plot_dir, exist_ok=True)

    # ---- Theme / rcParams (publication-ish) ----
    sns.set_theme(style="whitegrid", context="paper")  # 'paper' keeps fonts modest
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",   # safe default; switch to Arial if installed
        "axes.titlesize": 14,
        "axes.titleweight": "semibold",
        "axes.labelsize": 12,
        "xtick.labelsize": 10.5,
        "ytick.labelsize": 10.5,
        "legend.fontsize": 10,
        "axes.linewidth": 1.0,
        "grid.linewidth": 0.8,
    })

    # Long format once
    long_df = (
        proportions_df.reset_index()
        .melt(id_vars=["group"], value_vars=celltype_list, var_name="cell_type", value_name="proportion")
        .dropna()
    )
    long_df["group"] = pd.Categorical(long_df["group"], categories=existing_groups, ordered=True)

    global_max = long_df["proportion"].max() if (UNIFY_YLIM and not long_df.empty) else None

    # Pre-index Tukey/global for speed
    tukey_by_ct = {ct: df for ct, df in tukey_df.groupby("cell_type")} if (not tukey_df.empty and "cell_type" in tukey_df.columns) else {}
    global_by_ct = global_df.set_index("cell_type") if ("cell_type" in global_df.columns) else pd.DataFrame()

    def fmt_q(x):
        try:
            x = float(x)
        except Exception:
            return "NA"
        return "NA" if np.isnan(x) else f"{x:.2g}"

    def clean_name(s: str) -> str:
        return (s.replace("/", "_").replace(" ", "_").replace("+", "plus"))

    for ct in celltype_list:
        sub = long_df.loc[long_df["cell_type"] == ct, ["group", "proportion"]].copy()
        if sub.empty:
            continue

        # fetch global q-values from global_df
        if isinstance(global_by_ct, pd.DataFrame) and (not global_by_ct.empty) and (ct in global_by_ct.index):
            anova_q = global_by_ct.loc[ct, "anova_q_fdr"]
            kw_q    = global_by_ct.loc[ct, "kw_q_fdr"]
        else:
            anova_q = np.nan
            kw_q    = np.nan

        # ---- Figure: slightly wider for long labels ----
        fig, ax = plt.subplots(figsize=(7, 4.6), constrained_layout=True)

        # Grid aesthetics
        ax.grid(True, axis="y", color="#000000", alpha=0.10)
        ax.grid(False, axis="x")

        # Boxplot
        sns.boxplot(
            data=sub, x="group", y="proportion",
            order=existing_groups,
            palette=color,
            width=0.62,
            fliersize=0,
            linewidth=1.1,
            ax=ax
        )

        # Points: slightly smaller + white edge to separate from boxes
        sns.stripplot(
            data=sub, x="group", y="proportion",
            order=existing_groups,
            color="black",
            size=4.3,
            jitter=0.15,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.4,
            ax=ax
        )

        # Titles/labels
        ax.set_title(f"{ct}\nANOVA (q) = {fmt_q(anova_q)}   |   Kruskal (q) = {fmt_q(kw_q)}", pad=10)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion (per sample)")

        # x tick labels: less rotation, aligned
        ax.tick_params(axis="x", rotation=28)
        for lab in ax.get_xticklabels():
            lab.set_ha("right")

        # y-limits
        ymax_data = float(sub["proportion"].max()) if np.isfinite(sub["proportion"].max()) else 0.0
        ymax_base = float(global_max) if (global_max is not None and np.isfinite(global_max)) else ymax_data
        ymax = max(0.05, ymax_base * 1.15)
        ax.set_ylim(0, ymax)

        # Percent axis option
        if PERCENT_YAXIS:
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
            ax.set_ylabel("Proportion (%)")

        # ---- Tukey annotations ----
        tct = tukey_by_ct.get(ct, None)
        if tct is not None and not tct.empty and {"group1", "group2", "p_adj_tukey"}.issubset(tct.columns):
            ann = tct.loc[
                tct["group1"].isin(existing_groups) & tct["group2"].isin(existing_groups),
                ["group1", "group2", "p_adj_tukey"]
            ].copy()

            ann["p_adj_tukey"] = pd.to_numeric(ann["p_adj_tukey"], errors="coerce")
            ann = ann.dropna(subset=["p_adj_tukey"])
            ann["stars"] = ann["p_adj_tukey"].apply(p_to_stars)

            if ANNOTATE_ONLY_SIG:
                ann = ann.loc[ann["stars"] != ""].copy()

            ann = ann.sort_values("p_adj_tukey", ascending=True).head(MAX_PAIRS_TO_ANNOTATE)

            if not ann.empty:
                xmap = {g: i for i, g in enumerate(existing_groups)}
                y_min, y_max = ax.get_ylim()
                y_range = max(1e-6, y_max - y_min)

                # start a bit above top of data; step derived from range so it scales nicely
                y0 = max(ymax_data + 0.06 * y_range, 0.02)
                step = 0.08 * y_range
                h = 0.018 * y_range

                for k, (_, r) in enumerate(ann.iterrows()):
                    g1, g2 = r["group1"], r["group2"]
                    x1, x2 = xmap[g1], xmap[g2]
                    if x1 > x2:
                        x1, x2 = x2, x1
                    y = y0 + k * step
                    add_sig_bracket(ax, x1, x2, y=y, h=h, text=r["stars"], fontsize=12)

                # ensure enough headroom for brackets
                new_top = y0 + (len(ann) - 1) * step + h + 0.06 * y_range
                ax.set_ylim(y_min, max(y_max, new_top))

        sns.despine(ax=ax, left=False, bottom=False)

        safe = clean_name(ct)
        fout = os.path.join(plot_dir, f"Boxplot_{safe}_TukeyStars.{PLOT_FMT}")
        fig.savefig(fout, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved annotated per-celltype plots to: {plot_dir}")

print("\nALL DONE.")


### 5.10 Boxplot


In [ ]:
# ==============================================================================
# Boxplot Visualization - Cell Proportions by Group
# ==============================================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'al_proportion')
os.makedirs(prop_fig_dir, exist_ok=True)

# Calculate proportions
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group info
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

# Define cell types

celltype_list = [ct for ct in celltype_list if ct in proportions_df.columns]

# Melt for plotting
melted_df = pd.melt(
    proportions_df,
    id_vars=['group'],
    value_vars=celltype_list,
    var_name='Cell Type',
    value_name='Proportion'
)


# Faceted boxplot
g = sns.FacetGrid(
    melted_df, 
    col='Cell Type', 
    col_wrap=4, 
    height=3, 
    aspect=1.2, 
    sharey=False
)

g.map(sns.boxplot, 'group', 'Proportion', palette=color, order=color.keys())
g.map(sns.stripplot, 'group', 'Proportion', color='black', size=5, jitter=True, alpha=0.7)

# Rotate x labels
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

g.set_titles(col_template='{col_name}', size=10)
g.set_xlabels('')
g.fig.suptitle('Cell Type Proportions by Treatment Group', fontsize=14, y=1.02)

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_Boxplot_cell_proportions_by_group.pdf'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_Boxplot_cell_proportions_by_group.pdf')}")


---

## 6. DEG Analysis

**Comparisons vs NC (Control):**
1. RC148 vs NC
2. AK112 vs NC

### 6.1 Load Data


In [ ]:
# adata loaded above from Step3-sce_annotated.h5ad
adata


### 6.2 Check Distribution


In [ ]:
adata.raw.shape


In [ ]:
adata.obs['celltype'].value_counts()


### 6.3 Volcano Function


In [ ]:
# Ensure libraries imported
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

def plot_volcano_publication_quality(
    degs_df,
    title,
    subtitle,
    top_n_up=15,
    top_n_down=15,
    genes_to_highlight=None,
    lfc_threshold=1.0,
    pval_threshold=0.05,
    palette=None,
    savepath=None,
):
    """
    绘制一幅具有出版质量的火山图。
    
    Parameters:
    - degs_df: DEG DataFrame。
    - title: Main title。
    - subtitle: Subtitle。
    - top_n_up: Top N up genes。
    - top_n_down: Top N down genes。
    - genes_to_highlight: Gene list，这些基因将被特别标记。
    - lfc_threshold: Log2FC threshold。
    - pval_threshold: Adjusted p threshold。
    - palette: Color dict。
    """
    df = degs_df.copy()

    # --- 1. 数据准备 ---
    df['-log10_pvals_adj'] = -np.log10(df['pvals_adj'].astype(float) + 1e-300)
    df['status'] = 'Not significant'
    df.loc[(df['logfoldchanges'] > lfc_threshold) & (df['pvals_adj'] < pval_threshold), 'status'] = 'Up-regulated'
    df.loc[(df['logfoldchanges'] < -lfc_threshold) & (df['pvals_adj'] < pval_threshold), 'status'] = 'Down-regulated'
    
    # 颜色配置
    if palette is None:
        palette = {
            'Up-regulated': '#d62728',  # 更鲜亮的红色
            'Down-regulated': '#1f77b4', # 更鲜亮的蓝色
            'Not significant': '#cccccc'
        }

    # --- 2. 绘图 ---
    plt.style.use('seaborn-v0_8-whitegrid') # 使用一个干净的绘图风格
    fig, ax = plt.subplots(figsize=(12, 12))

    # 分别绘制点，以控制透明度和层级
    ax.scatter(
        df[df['status'] == 'Not significant']['logfoldchanges'],
        df[df['status'] == 'Not significant']['-log10_pvals_adj'],
        s=15, alpha=0.4, c=palette['Not significant'], label='Not significant', ec='none'
    )
    ax.scatter(
        df[df['status'] == 'Up-regulated']['logfoldchanges'],
        df[df['status'] == 'Up-regulated']['-log10_pvals_adj'],
        s=30, alpha=0.8, c=palette['Up-regulated'], label='Up-regulated', ec='none'
    )
    ax.scatter(
        df[df['status'] == 'Down-regulated']['logfoldchanges'],
        df[df['status'] == 'Down-regulated']['-log10_pvals_adj'],
        s=30, alpha=0.8, c=palette['Down-regulated'], label='Down-regulated', ec='none'
    )

    # --- 3. 核心改进：分离式标签选择 ---
    df['ranking_score'] = abs(df['logfoldchanges']) * df['-log10_pvals_adj']
    
    up_genes = df[df['status'] == 'Up-regulated'].sort_values('ranking_score', ascending=False).head(top_n_up)
    down_genes = df[df['status'] == 'Down-regulated'].sort_values('ranking_score', ascending=False).head(top_n_down)
    
    genes_to_label_df = pd.concat([up_genes, down_genes])
    
    # 如果有指定要高亮的基因，也加入标签列表
    if genes_to_highlight:
        highlight_df = df[df['names'].isin(genes_to_highlight)]
        genes_to_label_df = pd.concat([genes_to_label_df, highlight_df]).drop_duplicates(subset=['names'])

    texts = []
    for _, row in genes_to_label_df.iterrows():
        texts.append(ax.text(row['logfoldchanges'], row['-log10_pvals_adj'], row['names'], fontsize=12))

    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle='-', color='grey', lw=0.5),
                force_points=(0.2, 0.5),
                force_text=(0.5, 1.0))

    # --- 4. 添加注释和美化 ---
    # 阈值线
    ax.axhline(y=-np.log10(pval_threshold), color='grey', linestyle='--', linewidth=1)
    ax.axvline(x=lfc_threshold, color='grey', linestyle='--', linewidth=1)
    ax.axvline(x=-lfc_threshold, color='grey', linestyle='--', linewidth=1)
    
    # 统计数量注释
    num_up = (df['status'] == 'Up-regulated').sum()
    num_down = (df['status'] == 'Down-regulated').sum()
    ax.text(0.02, 0.98, f'Up: {num_up}\nDown: {num_down}', transform=ax.transAxes,
            fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.5, ec='none'))

    # 标题和轴标签
    fig.suptitle(title, fontsize=20, weight='bold')
    ax.set_title(subtitle, fontsize=14, pad=10)
    ax.set_xlabel("Log2 Fold Change", fontsize=14)
    ax.set_ylabel("-log10(Adjusted P-value)", fontsize=14)

    # 图例
    ax.legend(loc='upper right', frameon=False, fontsize=12)
    
    # 移除顶部和右侧的轴线
    sns.despine(ax=ax)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # 为主标题留出空间
    if savepath:
        plt.savefig(savepath, dpi=300)
    plt.show()


### 6.4 Run All Comparisons


In [ ]:
# ============================================================
# Full DEG + Volcano + Enrichr(ORA) + Enrichment dotplot pipeline
# ============================================================

# ---- Imports ----
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

# ------------------------------------------------------------
# 0) Utilities
# ------------------------------------------------------------
def safe_filename(s: str) -> str:
    return re.sub(r'[^0-9a-zA-Z\-\_\.]', '_', str(s))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

# ------------------------------------------------------------
# 1) Enrichr dotplot
# ------------------------------------------------------------
def enrichr_dotplot(enr_results: pd.DataFrame, title: str, top_n: int = 15, savepath: str | None = None):
    """
    Plot Enrichr ORA results (gseapy.enrichr(...).results) as dotplot.

    Expected columns (typical Enrichr):
      - 'Term'
      - 'Adjusted P-value'
      - 'Combined Score'
      - 'Overlap' (e.g., "12/200")

    This function is defensive across minor column name differences.
    """
    if enr_results is None or not isinstance(enr_results, pd.DataFrame) or enr_results.empty:
        return None

    df = enr_results.copy()

    # Find columns robustly
    term_col = 'Term' if 'Term' in df.columns else None
    if term_col is None:
        # fallback: first column that looks like term/name
        for c in df.columns:
            if c.lower() in ('term', 'pathway', 'gene_set', 'name'):
                term_col = c
                break
    if term_col is None:
        term_col = df.columns[0]

    q_col = 'Adjusted P-value' if 'Adjusted P-value' in df.columns else None
    if q_col is None:
        for c in df.columns:
            lc = c.lower()
            if ('adjust' in lc or 'adj' in lc) and ('p' in lc):
                q_col = c
                break
    if q_col is None:
        raise ValueError(f"Cannot find adjusted p-value column in Enrichr results: {df.columns.tolist()}")

    score_col = 'Combined Score' if 'Combined Score' in df.columns else None
    if score_col is None:
        for c in df.columns:
            if 'score' in c.lower():
                score_col = c
                break

    overlap_col = 'Overlap' if 'Overlap' in df.columns else None

    # Prep plot table
    df[q_col] = df[q_col].astype(float)
    df['-log10(q)'] = -np.log10(df[q_col] + 1e-300)

    # hits from overlap
    if overlap_col is not None:
        try:
            hits = df[overlap_col].astype(str).str.split('/', expand=True)[0].astype(float)
            df['Hits'] = hits
        except Exception:
            df['Hits'] = 1.0
    else:
        df['Hits'] = 1.0

    df = df.sort_values(q_col, ascending=True).head(top_n).copy()
    df = df.iloc[::-1]  # reverse for top term on top

    # Plot
    plt.figure(figsize=(10, max(4, 0.38 * len(df) + 1.7)))
    ax = plt.gca()

    cvals = df[score_col] if score_col is not None else df['-log10(q)']
    sca = ax.scatter(
        df['-log10(q)'],
        df[term_col],
        s=40 + 12 * df['Hits'],
        c=cvals,
        cmap='viridis',
        alpha=0.92,
        edgecolor='none'
    )

    ax.set_title(title, fontsize=14, weight='bold')
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_ylabel('')

    cbar = plt.colorbar(sca, ax=ax, pad=0.02)
    cbar.set_label(score_col if score_col is not None else '-log10(q)')

    sns.despine(ax=ax, left=False, bottom=False)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300)
        plt.close()
        return savepath
    else:
        plt.show()
        return None

# ------------------------------------------------------------
# 2) Core runner
# ------------------------------------------------------------
def run_deg_volcano_enrichr_all(
    adata,
    comparisons,
    main_type_col='celltype',
    compare_col='group',
    lfc_threshold=0.5,
    pval_threshold=0.05,
    gene_sets=('GO_Biological_Process_2023',),
    organism='mouse',
    outdir='./DEG_Enrichr_all_comparisons',
    min_cells=20,
    top_n_terms=15,
):
    """
    For each comparison (group1 vs group2) and each celltype:
      1) DEG via scanpy rank_genes_groups (wilcoxon)
      2) Save DEG table
      3) Volcano plot (requires plot_volcano_publication_quality defined)
      4) Enrichr ORA on significant Up and Down lists
      5) Save Enrichr tables + dotplots

    Returns:
      all_results: nested dict [comp_name][celltype] -> dict with degs/sig/enrichr results (DataFrames)
    """
    ensure_dir(outdir)

    all_celltypes = pd.Index(adata.obs[main_type_col].unique()).tolist()
    all_results = {}

    # Record config
    cfg = dict(
        comparisons=list(comparisons),
        main_type_col=main_type_col,
        compare_col=compare_col,
        lfc_threshold=lfc_threshold,
        pval_threshold=pval_threshold,
        gene_sets=list(gene_sets),
        organism=organism,
        outdir=outdir,
        min_cells=min_cells,
        top_n_terms=top_n_terms,
    )
    pd.Series(cfg).to_csv(os.path.join(outdir, "run_config.csv"))

    for group1, group2 in comparisons:
        comp_name = f"{group1}_vs_{group2}"
        safe_comp = safe_filename(comp_name)

        print(f"\n{'='*70}")
        print(f"Running: {comp_name}")
        print(f"{'='*70}")

        comp_dir = ensure_dir(os.path.join(outdir, safe_comp))
        comp_results = {}

        for celltype in all_celltypes:
            print(f"Processing celltype: {celltype}")
            safe_ct = safe_filename(celltype)

            # subset by celltype + groups
            adata_sub = adata[adata.obs[main_type_col] == celltype].copy()
            adata_sub = adata_sub[adata_sub.obs[compare_col].isin([group1, group2])].copy()

            if adata_sub.n_obs < min_cells:
                print(f"  Skipping: only {adata_sub.n_obs} cells (<{min_cells})")
                continue

            # DEG
            key = f"rank_genes_{safe_comp}"
            try:
                sc.tl.rank_genes_groups(
                    adata_sub,
                    groupby=compare_col,
                    groups=[group1],
                    reference=group2,
                    method='wilcoxon',
                    key_added=key
                )
            except Exception as e:
                print(f"  DEG failed: {e}")
                continue

            degs = sc.get.rank_genes_groups_df(adata_sub, group=group1, key=key)

            # Defensive: ensure expected columns exist
            required_cols = {'names', 'logfoldchanges', 'pvals_adj'}
            missing = required_cols - set(degs.columns)
            if missing:
                print(f"  DEG table missing columns {missing}; got {degs.columns.tolist()}")
                continue

            # Significant set
            sig = degs[(degs['pvals_adj'] < pval_threshold) &
                       (degs['logfoldchanges'].abs() > lfc_threshold)].copy()

            print(f"  Cells: {adata_sub.n_obs}")
            print(f"  Significant DEGs: {len(sig)}")

            # Save DEG tables
            deg_csv = os.path.join(comp_dir, f"{safe_ct}_DEG.csv")
            sig_csv = os.path.join(comp_dir, f"{safe_ct}_DEG_sig.csv")
            degs.to_csv(deg_csv, index=False)
            sig.to_csv(sig_csv, index=False)

            # Volcano
            volc_png = os.path.join(comp_dir, f"{safe_ct}_volcano.png")
            try:
                plot_volcano_publication_quality(
                    degs,
                    title=f"DEG: {celltype} ({group1} vs {group2})",
                    subtitle=f"Top genes in {celltype}",
                    lfc_threshold=lfc_threshold,
                    pval_threshold=pval_threshold,
                    savepath=volc_png
                )
            except Exception as e:
                print(f"  Volcano failed: {e}")

            # Enrichr ORA: up/down lists
            up = sig.loc[sig['logfoldchanges'] > lfc_threshold, 'names'].dropna().astype(str).tolist()
            down = sig.loc[sig['logfoldchanges'] < -lfc_threshold, 'names'].dropna().astype(str).tolist()

            enrichr_tables = {}

            # Up
            if len(up) > 5:
                try:
                    enr_up = gp.enrichr(
                        gene_list=up,
                        gene_sets=list(gene_sets),
                        organism=organism,
                        outdir=None
                    )
                    if enr_up is not None and enr_up.results is not None and not enr_up.results.empty:
                        up_res = enr_up.results.copy()
                        up_csv = os.path.join(comp_dir, f"{safe_ct}_Enrichr_up.csv")
                        up_res.to_csv(up_csv, index=False)

                        up_fig = os.path.join(comp_dir, f"{safe_ct}_Enrichr_up_dotplot.png")
                        enrichr_dotplot(
                            up_res,
                            title=f"Enrichr Up: {celltype} ({group1} vs {group2})",
                            top_n=top_n_terms,
                            savepath=up_fig
                        )

                        enrichr_tables['up'] = up_res
                        print(f"  Enrichr up: {len(up_res)} terms")
                    else:
                        print("  Enrichr up: empty results")
                except Exception as e:
                    print(f"  Enrichr up failed: {e}")
            else:
                print(f"  Enrichr up skipped: only {len(up)} genes")

            # Down
            if len(down) > 5:
                try:
                    enr_down = gp.enrichr(
                        gene_list=down,
                        gene_sets=list(gene_sets),
                        organism=organism,
                        outdir=None
                    )
                    if enr_down is not None and enr_down.results is not None and not enr_down.results.empty:
                        down_res = enr_down.results.copy()
                        down_csv = os.path.join(comp_dir, f"{safe_ct}_Enrichr_down.csv")
                        down_res.to_csv(down_csv, index=False)

                        down_fig = os.path.join(comp_dir, f"{safe_ct}_Enrichr_down_dotplot.png")
                        enrichr_dotplot(
                            down_res,
                            title=f"Enrichr Down: {celltype} ({group1} vs {group2})",
                            top_n=top_n_terms,
                            savepath=down_fig
                        )

                        enrichr_tables['down'] = down_res
                        print(f"  Enrichr down: {len(down_res)} terms")
                    else:
                        print("  Enrichr down: empty results")
                except Exception as e:
                    print(f"  Enrichr down failed: {e}")
            else:
                print(f"  Enrichr down skipped: only {len(down)} genes")

            comp_results[celltype] = {
                'degs': degs,
                'sig_degs': sig,
                'up_genes': up,
                'down_genes': down,
                'enrichr': enrichr_tables,
                'paths': {
                    'degs_csv': deg_csv,
                    'sig_csv': sig_csv,
                    'volcano_png': volc_png,
                }
            }

        all_results[comp_name] = comp_results

    print("\nAll comparisons complete!")
    return all_results

# ------------------------------------------------------------
# 3) Run (your parameters)
# ------------------------------------------------------------

comparisons = [
    ('R301', 'PBS'),
]

main_type_col = 'celltype'
compare_col = 'group'
lfc_threshold = 0.5
pval_threshold = 0.05
gene_sets = ['GO_Biological_Process_2023']
organism = 'mouse'
outdir = './DEG_Enrichr_all_comparisons'

all_results = run_deg_volcano_enrichr_all(
    adata=adata,
    comparisons=comparisons,
    main_type_col=main_type_col,
    compare_col=compare_col,
    lfc_threshold=lfc_threshold,
    pval_threshold=pval_threshold,
    gene_sets=gene_sets,
    organism=organism,
    outdir=outdir,
    min_cells=20,
    top_n_terms=15,
)

# Optional: quick summary
summary_rows = []
for comp, d in all_results.items():
    for ct, res in d.items():
        summary_rows.append({
            'comparison': comp,
            'celltype': ct,
            'n_deg_total': len(res['degs']),
            'n_deg_sig': len(res['sig_degs']),
            'n_up': len(res['up_genes']),
            'n_down': len(res['down_genes']),
            'has_enrichr_up': 'up' in res['enrichr'],
            'has_enrichr_down': 'down' in res['enrichr'],
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(outdir, "summary.csv"), index=False)
summary_df.head()


### 6.6 Summarize


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Config
# ----------------------------
OUTDIR = "./DEG_Enrichr_all_comparisons"  # 改成你的实际输出目录
PVAL_THRESHOLD = 0.05
TOP_TERMS_PER_CONDITION = 15     # 每个 celltype×comparison×direction 取前N个term用于热图
TOP_TERMS_FREQUENCY = 25         # 频次柱状图展示前N个term
FIG_DPI = 300

# ----------------------------
# Helpers
# ----------------------------
def _read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def _infer_celltype_from_filename(fname):
    # expects: <celltype>_DEG_sig.csv or <celltype>_Enrichr_up.csv
    base = os.path.basename(fname)
    for suffix in ["_DEG_sig.csv", "_DEG.csv", "_Enrichr_up.csv", "_Enrichr_down.csv"]:
        if base.endswith(suffix):
            return base[: -len(suffix)]
    return os.path.splitext(base)[0]

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------
# 1) Summarize DEG counts
# ----------------------------
def summarize_deg_counts(outdir):
    rows = []

    comp_dirs = [d for d in glob.glob(os.path.join(outdir, "*")) if os.path.isdir(d)]
    for comp_dir in sorted(comp_dirs):
        comp_name = os.path.basename(comp_dir)

        sig_files = glob.glob(os.path.join(comp_dir, "*_DEG_sig.csv"))
        # fallback: if sig not saved, we can compute from DEG.csv (less ideal)
        if not sig_files:
            sig_files = glob.glob(os.path.join(comp_dir, "*_DEG.csv"))

        for f in sorted(sig_files):
            celltype = _infer_celltype_from_filename(f)
            df = _read_csv_safe(f)
            if df is None or df.empty:
                continue

            # Determine whether it's sig table or full DEG table
            is_sig = f.endswith("_DEG_sig.csv")

            # Need at least logfoldchanges, pvals_adj if full DEG
            if not is_sig:
                required = {"logfoldchanges", "pvals_adj"}
                if not required.issubset(df.columns):
                    continue
                df_sig = df[(df["pvals_adj"] < PVAL_THRESHOLD)].copy()
            else:
                df_sig = df.copy()

            if "logfoldchanges" not in df_sig.columns:
                continue

            n_sig = len(df_sig)
            n_up = int((df_sig["logfoldchanges"] > 0).sum())
            n_down = int((df_sig["logfoldchanges"] < 0).sum())

            rows.append({
                "comparison": comp_name,
                "celltype": celltype,
                "n_sig": n_sig,
                "n_up": n_up,
                "n_down": n_down,
                "sig_source": "DEG_sig" if is_sig else "DEG_filtered_by_padj"
            })

    res = pd.DataFrame(rows)
    return res

def plot_deg_counts_heatmaps(deg_counts_df, outdir_fig):
    ensure_dir(outdir_fig)

    if deg_counts_df is None or deg_counts_df.empty:
        print("No DEG summary data to plot.")
        return

    for metric, title in [
        ("n_sig", "Significant DEG counts (Up+Down)"),
        ("n_up", "Upregulated DEG counts"),
        ("n_down", "Downregulated DEG counts"),
    ]:
        mat = deg_counts_df.pivot_table(
            index="celltype", columns="comparison", values=metric, aggfunc="max", fill_value=0
        )

        plt.figure(figsize=(max(8, 0.6 * mat.shape[1] + 3), max(6, 0.35 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="mako", linewidths=0.3, linecolor="white")
        ax.set_title(title, fontsize=14, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        path = os.path.join(outdir_fig, f"DEG_counts_heatmap_{metric}.png")
        plt.savefig(path, dpi=FIG_DPI)
        plt.close()

    # Also save a long-format barplot-friendly table
    deg_counts_df.to_csv(os.path.join(outdir_fig, "summary_deg_counts.csv"), index=False)
    print(f"Saved DEG summaries to: {outdir_fig}")

# ----------------------------
# 2) Summarize Enrichr results (Up/Down)
# ----------------------------
def summarize_enrichr(outdir):
    """
    Returns long table:
      comparison, celltype, direction(up/down), term, q, combined_score, overlap, genes...
    """
    rows = []
    comp_dirs = [d for d in glob.glob(os.path.join(outdir, "*")) if os.path.isdir(d)]

    for comp_dir in sorted(comp_dirs):
        comp_name = os.path.basename(comp_dir)

        for direction in ["up", "down"]:
            files = glob.glob(os.path.join(comp_dir, f"*_Enrichr_{direction}.csv"))
            for f in sorted(files):
                celltype = _infer_celltype_from_filename(f)
                df = _read_csv_safe(f)
                if df is None or df.empty:
                    continue

                # standard enrichr columns
                # Must have term + adjusted pvalue; be defensive
                term_col = "Term" if "Term" in df.columns else df.columns[0]

                q_col = "Adjusted P-value" if "Adjusted P-value" in df.columns else None
                if q_col is None:
                    for c in df.columns:
                        lc = c.lower()
                        if ("adjust" in lc or "adj" in lc) and ("p" in lc):
                            q_col = c
                            break
                if q_col is None:
                    continue

                score_col = "Combined Score" if "Combined Score" in df.columns else None
                if score_col is None:
                    for c in df.columns:
                        if "score" in c.lower():
                            score_col = c
                            break

                overlap_col = "Overlap" if "Overlap" in df.columns else None
                genes_col = "Genes" if "Genes" in df.columns else None

                tmp = df.copy()
                tmp = tmp.rename(columns={term_col: "term", q_col: "q"})
                tmp["comparison"] = comp_name
                tmp["celltype"] = celltype
                tmp["direction"] = direction

                if score_col and score_col in tmp.columns:
                    tmp = tmp.rename(columns={score_col: "combined_score"})
                else:
                    tmp["combined_score"] = np.nan

                if overlap_col and overlap_col in tmp.columns:
                    tmp = tmp.rename(columns={overlap_col: "overlap"})
                else:
                    tmp["overlap"] = np.nan

                if genes_col and genes_col in tmp.columns:
                    tmp = tmp.rename(columns={genes_col: "genes"})
                else:
                    tmp["genes"] = np.nan

                keep = ["comparison", "celltype", "direction", "term", "q", "combined_score", "overlap", "genes"]
                tmp = tmp[keep].copy()

                # ensure numeric q
                tmp["q"] = pd.to_numeric(tmp["q"], errors="coerce")
                tmp["combined_score"] = pd.to_numeric(tmp["combined_score"], errors="coerce")

                rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["comparison","celltype","direction","term","q","combined_score","overlap","genes"])

    res = pd.concat(rows, ignore_index=True)
    return res

def plot_enrichr_frequency_and_heatmap(enr_long_df, outdir_fig):
    ensure_dir(outdir_fig)

    if enr_long_df is None or enr_long_df.empty:
        print("No Enrichr summary data to plot.")
        return

    # Save long table
    enr_long_df.to_csv(os.path.join(outdir_fig, "summary_enrichr_long.csv"), index=False)

    # Add -log10q
    df = enr_long_df.copy()
    df = df.dropna(subset=["q", "term"])
    df["-log10q"] = -np.log10(df["q"].astype(float) + 1e-300)

    for direction in ["up", "down"]:
        d = df[df["direction"] == direction].copy()
        if d.empty:
            continue

        # ---- Frequency barplot: how many (celltype×comparison) a term appears with q<PVAL_THRESHOLD
        d_sig = d[d["q"] < PVAL_THRESHOLD].copy()
        if d_sig.empty:
            continue

        freq = (d_sig.groupby("term")
                    .size()
                    .sort_values(ascending=False)
                    .head(TOP_TERMS_FREQUENCY))

        plt.figure(figsize=(10, max(5, 0.35 * len(freq) + 1.5)))
        ax = sns.barplot(x=freq.values, y=freq.index, color="#4C78A8")
        ax.set_title(f"Enrichr term frequency (q<{PVAL_THRESHOLD}) [{direction}]", fontsize=14, weight="bold")
        ax.set_xlabel("Count of (celltype × comparison) where term is significant")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_frequency_{direction}.png"), dpi=FIG_DPI)
        plt.close()

        # ---- Heatmap: term × condition (celltype|comparison) values = max(-log10q) among duplicates
        # pick top terms overall (by frequency) to keep heatmap readable
        top_terms = freq.index.tolist()

        d2 = d_sig[d_sig["term"].isin(top_terms)].copy()
        d2["condition"] = d2["celltype"].astype(str) + " | " + d2["comparison"].astype(str)

        mat = d2.pivot_table(
            index="term", columns="condition", values="-log10q", aggfunc="max", fill_value=0
        )

        # sort columns by comparison then celltype for readability
        # (optional) keep as-is; pivot_table column order is lexical
        plt.figure(figsize=(max(10, 0.25 * mat.shape[1] + 4), max(6, 0.35 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="rocket_r", linewidths=0.2, linecolor="white")
        ax.set_title(f"Enrichr heatmap: -log10(q) [{direction}] (top terms by frequency)", fontsize=14, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_heatmap_{direction}.png"), dpi=FIG_DPI)
        plt.close()

    print(f"Saved Enrichr summaries to: {outdir_fig}")

# ----------------------------
# 3) (Optional) Term heatmap per comparison (more focused)
# ----------------------------
def plot_enrichr_heatmap_per_comparison(enr_long_df, outdir_fig, direction="up"):
    """
    For each comparison separately:
      term × celltype heatmap, using top terms within that comparison.
    """
    ensure_dir(outdir_fig)
    if enr_long_df is None or enr_long_df.empty:
        return

    df = enr_long_df.dropna(subset=["q", "term"]).copy()
    df["-log10q"] = -np.log10(df["q"].astype(float) + 1e-300)
    df = df[(df["direction"] == direction) & (df["q"] < PVAL_THRESHOLD)].copy()
    if df.empty:
        return

    for comp, dcomp in df.groupby("comparison"):
        # pick top terms inside this comparison (by median -log10q)
        term_rank = (dcomp.groupby("term")["-log10q"].median().sort_values(ascending=False))
        top_terms = term_rank.head(TOP_TERMS_PER_CONDITION).index.tolist()

        d2 = dcomp[dcomp["term"].isin(top_terms)]
        mat = d2.pivot_table(index="term", columns="celltype", values="-log10q", aggfunc="max", fill_value=0)

        plt.figure(figsize=(max(10, 0.35 * mat.shape[1] + 3), max(5, 0.4 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="rocket_r", linewidths=0.2, linecolor="white")
        ax.set_title(f"{comp}: Enrichr -log10(q) heatmap [{direction}]", fontsize=13, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_heatmap_{direction}__{comp}.png"), dpi=FIG_DPI)
        plt.close()

# ----------------------------
# 4) Run everything
# ----------------------------
FIG_OUT = ensure_dir(os.path.join(OUTDIR, "_summary_figures"))

deg_counts = summarize_deg_counts(OUTDIR)
deg_counts.to_csv(os.path.join(FIG_OUT, "summary_deg_counts.csv"), index=False)
plot_deg_counts_heatmaps(deg_counts, FIG_OUT)

enr_long = summarize_enrichr(OUTDIR)
enr_long.to_csv(os.path.join(FIG_OUT, "summary_enrichr_long.csv"), index=False)
plot_enrichr_frequency_and_heatmap(enr_long, FIG_OUT)

# Optional: per-comparison term×celltype heatmaps (often更好读)
plot_enrichr_heatmap_per_comparison(enr_long, FIG_OUT, direction="up")
plot_enrichr_heatmap_per_comparison(enr_long, FIG_OUT, direction="down")

print("Done. Summary figures saved to:", FIG_OUT)
